# Shared Loading & Functions

In [ ]:
import os, textwrap
from types import SimpleNamespace
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

save = True
MAIN_FIGURES_ONLY = True  # True: only the main figures (included sample); False: every figure
COMPARE_PILOT = True     # True: only pilot (left, 'Pilot') vs. replication (right, 'Replication') figures, saved under COMPARISON_ROOT
USE_PILOT = False         # True: run everything on the pilot data instead of the replication (figures save under the pilot's figures/)

plt.rcParams.update({'figure.dpi': 200, 'savefig.dpi': 200,
                     'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 13, 'axes.titlesize': 15, 'axes.labelsize': 14, 'figure.titlesize': 16,
                     'xtick.labelsize': 12, 'ytick.labelsize': 12, 'legend.fontsize': 12, 'legend.title_fontsize': 12})

PILOT_ROOT = '../pilot/episodic-choice-task'
DATA_ROOT = PILOT_ROOT if USE_PILOT else '..'
COMPARISON_ROOT = '../figures/pilot_comparison'  # with COMPARE_PILOT, figures save under here instead
assert not (USE_PILOT and COMPARE_PILOT), 'COMPARE_PILOT already puts the pilot next to the replication; turn off USE_PILOT'

MEM_ORDER = ['low', 'mid', 'high']
MEM_LABELS = ['Low', 'Mid', 'High']
# memorability is always light -> dark (low -> high); pick the hue per measure
MEM_PALETTES = {
    'blue':   dict(zip(MEM_ORDER, ['#9ecae1', '#3182bd', '#08519c'])),
    'purple': dict(zip(MEM_ORDER, ['#bcbddc', '#756bb1', '#54278f'])),
    'green':  dict(zip(MEM_ORDER, ['#a1d99b', '#31a354', '#006d2c'])),
}
VALUE_PALETTE = {0.0: '#fdae6b', 1.0: '#cc4c02'}  # $0 light orange, $1 burnt orange
GRAY_PALETTE = {'low': '#bdbdbd', 'mid': '#969696', 'high': '#525252'}  # incorrect trials, light -> dark


def marker_style(n_lines):
    """Black-ringed markers; fewer mean lines on the plot -> bigger markers."""
    return dict(markeredgecolor='black', markeredgewidth=1, markersize={1: 12, 2: 10, 3: 8}[n_lines])


SMALL_FIGSIZE = (3.1, 4.2)  # single-panel size for two/three-condition plots ($0 vs $1, low vs high, ...)

# every figure spec starts from these; a spec only lists what it changes.
# Main figures also list ylim=(bottom, top) right after their fname so it's easy to edit (None = automatic).
FIG_DEFAULTS = dict(
    included_figsize=SMALL_FIGSIZE,
    ylim=(None, None),
    ytick_step=0.1,          # y ticks on multiples of this (probabilities, RT in s)
    chance_color='gray',
    errorbar=('ci', 95),     # error bars: 95% CI of the per-subject means ('se' for standard error)
    **marker_style(1),       # override with **marker_style(2) / (3) when a plot has 2 / 3 lines
)

"""COLUMNS
'experiment_id','participant_id', 'is_choice_trial','is_attention_check','rt','response', 'correct','trial_number','block_index','old_trial',
'memorability_bin','encoding_trial','delay','old_side','old_value','left_image_name','left_image_path','left_memorability','left_value',
'left_is_old','right_image_name','right_image_path','right_memorability','right_value','right_is_old','chosen_side','chosen_image_name',
'chosen_image_path','chosen_value','reward','repeat_source_was_chosen','response_key','choice_missed','old_chosen','optimal_old_choice','final_bonus'
"""

In [ ]:
### LOADING & EXCLUSIONS

def load_data(label, data_root=DATA_ROOT):
    fname = 'episodic_choice_data.csv' if label == 'main' else f'episodic_choice_data-{label}.csv'
    df = pd.read_csv(f'{data_root}/data/{fname}')
    figures_dir = f'{COMPARISON_ROOT}/{label}' if COMPARE_PILOT else f'{data_root}/figures/{label}'
    os.makedirs(figures_dir, exist_ok=True)
    return df, figures_dir


def finish_fig(figures_dir, fname, show=None):
    """Save (if save) and display; non-main figures are closed unshown when MAIN_FIGURES_ONLY."""
    plt.tight_layout()
    if save:
        plt.savefig(os.path.join(figures_dir, fname))
    if show if show is not None else not MAIN_FIGURES_ONLY:
        plt.show()
    else:
        plt.close()


def find_ai_pids(df):
    """Participants who pressed the hidden key on attention checks (white-text AI check).
    Old (episodic_choice_v3) data used the space bar; everything else uses 'x'."""
    attention_df = df.query('is_attention_check == True')
    is_old_format = attention_df['experiment_id'] == 'episodic_choice_v3'
    hidden_df = pd.concat([
        attention_df[is_old_format].query('response_key == " "'),
        attention_df[~is_old_format].query('response_key == "x" or response == "x"'),
    ])
    print(f'{len(hidden_df)} hidden-key (space or x) responses on attention checks')
    print(hidden_df[['participant_id', 'trial_number', 'response', 'response_key', 'correct']])
    ai_pids = hidden_df['participant_id'].unique().tolist()

    if ai_pids:
        choice_df = df[df['participant_id'].isin(ai_pids) & (df['is_choice_trial'] == True)]
        if not COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
            sns.displot(x='rt', data=choice_df, col='participant_id')
            plt.suptitle(f'SUSPECTED AI RESPONSE TIMES\n(n={len(ai_pids)})')
            plt.tight_layout()
            plt.show() if not MAIN_FIGURES_ONLY else plt.close()
        print(f"Suspected AI optimal old choice rate: {pd.to_numeric(choice_df['optimal_old_choice'], errors='coerce').mean():.3f}")
    return ai_pids


def find_failed_attention_pids(df, figures_dir, threshold=1):
    """Participants below threshold accuracy on the attention checks (default: failed any); also plots per-subject accuracy."""
    plot_df = df.query('is_attention_check == True').groupby('participant_id', as_index=False).agg({'correct': 'mean'})
    plot_df = plot_df.sort_values('correct').reset_index(drop=True)
    plot_df['x'] = np.arange(len(plot_df))
    failed_pids = plot_df.query('correct < @threshold')['participant_id'].tolist()
    if COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
        return failed_pids

    colors = ['tomato' if pid in failed_pids else 'gray' for pid in plot_df['participant_id']]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(plot_df['x'], plot_df['correct'], color=colors)
    ax.set_title(f"Attention Check Performance\n(n={plot_df['participant_id'].nunique()})")
    ax.set_ylabel('Proportion correct')
    ax.set_xlabel('Participant')
    ax.set_xticks(plot_df['x'])
    ax.set_xticklabels(plot_df['participant_id'], rotation=30, ha='right', fontsize=3)
    ax.set_ylim(0, 1.1)
    finish_fig(figures_dir, 'attention_checks.png')
    return failed_pids


def find_incomplete_pids(df, trial_query, min_trials):
    counts = df.query(trial_query).groupby('participant_id').size()
    return counts[counts < min_trials].index.tolist()


def find_high_miss_pids(df, threshold=0.2):
    """Preregistered: no response on > threshold of all response trials (value-report prompts count in the direct task)."""
    response_df = df.query('is_choice_trial == True')
    no_col = pd.Series(False, index=response_df.index)
    is_value_test = response_df.get('is_value_test_trial', no_col) == True
    is_missed = lambda s: s.map(lambda v: str(v).lower() in ('true', '1', '1.0'))
    missed = ((is_missed(response_df.get('value_test_missed', no_col)) & is_value_test) |
              (is_missed(response_df['choice_missed']) & ~is_value_test))
    miss_rate = missed.groupby(response_df['participant_id']).mean()
    return miss_rate[miss_rate > threshold].index.tolist()


def binom_test_by_subject(data, col):
    """One-sample binomial test per subject against chance (0.5), one-tailed."""
    rows = []
    for pid, vals in data.dropna(subset=[col]).groupby('participant_id')[col]:
        p = stats.binomtest(int(vals.sum()), len(vals), 0.5, alternative='greater').pvalue
        rows.append({'participant_id': pid, col: vals.mean(), 'pval': p, 'significant': p < 0.05})
    return pd.DataFrame(rows)


def plot_overall_performance(df, perf_query, perf_col, perf_label, ai_pids, failed_pids, figures_dir):
    """Per-subject performance bars (non-significant first); returns the significantly-above-chance pids."""
    data = df.query(perf_query).copy()
    data[perf_col] = pd.to_numeric(data[perf_col], errors='coerce')
    plot_df = binom_test_by_subject(data, perf_col)
    plot_df = plot_df.sort_values(['significant', perf_col]).reset_index(drop=True)
    plot_df['x'] = np.arange(len(plot_df))
    print(f"{int(plot_df['significant'].sum())} out of {len(plot_df)} participants significantly above chance "
          f"on {perf_col} (one-sample binomial test, p < 0.05, one-tailed)")
    sig_pids = plot_df.query('significant')['participant_id'].tolist()
    if COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
        return sig_pids

    def bar_color(row):
        if row['participant_id'] in ai_pids:
            return 'mediumpurple'
        elif row['participant_id'] in failed_pids:
            return 'tomato'
        elif row['significant']:
            return 'steelblue'
        return 'gray'

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(plot_df['x'], plot_df[perf_col], color=[bar_color(row) for _, row in plot_df.iterrows()])
    ax.set_title(f"Overall {perf_label}\n(n={plot_df['participant_id'].nunique()})")
    ax.set_xlabel('Participant')
    ax.set_ylabel(perf_label)
    ax.set_xticks(plot_df['x'])
    ax.set_xticklabels(plot_df['participant_id'], rotation=30, ha='right', fontsize=3)
    ax.axhline(y=0.5, linestyle='dashed', color='red')
    ax.set_ylim(0, 1)

    # vertical line between non-significant and significant participants
    n_not_sig = int((~plot_df['significant']).sum())
    if 0 < n_not_sig < len(plot_df):
        ax.axvline(x=n_not_sig - 0.5, color='black', linestyle='--', linewidth=1.5)

    ax.legend(handles=[
        Patch(facecolor='steelblue', label='Significant above chance'),
        Patch(facecolor='gray', label='Not significant'),
        Patch(facecolor='tomato', label='Failed attention check'),
        Patch(facecolor='mediumpurple', label='Suspected AI'),
    ], loc='upper left', fontsize=8)
    finish_fig(figures_dir, 'optimal_subject.png')
    return sig_pids


def exclude(exp, pids):
    """Add pids to the experiment's exclusions and recompute the included sample."""
    exp.excluded_pids = exp.excluded_pids | set(pids)
    exp.included_pids = [pid for pid in exp.sig_pids if pid not in exp.excluded_pids]


def setup_experiment(label, perf_query='old_trial == 1', perf_col='optimal_old_choice',
                     perf_label='Old Trial Optimal Performance', min_trials=None, data_root=DATA_ROOT):
    """Load one task's data and run the shared exclusion checks.

    Included participants are significantly above chance on perf_col (one-tailed binomial test),
    are not suspected AI, pass every attention check, respond on at least 80% of response trials, and
    (if min_trials is given) have at least min_trials trials matching perf_query. Same criteria as
    scripts/count_participants.py."""
    source = 'Pilot' if data_root == PILOT_ROOT else 'Replication'
    print(f'===== {label} ({source}) =====')
    df, figures_dir = load_data(label, data_root)
    ai_pids = find_ai_pids(df)
    failed_pids = find_failed_attention_pids(df, figures_dir)
    incomplete_pids = find_incomplete_pids(df, perf_query, min_trials) if min_trials else []
    high_miss_pids = find_high_miss_pids(df)
    print(f'{len(high_miss_pids)} participants with > 20% missed responses')
    sig_pids = plot_overall_performance(df, perf_query, perf_col, perf_label, ai_pids, failed_pids, figures_dir)

    exp = SimpleNamespace(label=label, source=source, df=df, figures_dir=figures_dir, ai_pids=ai_pids, failed_pids=failed_pids,
                          incomplete_pids=incomplete_pids, high_miss_pids=high_miss_pids, sig_pids=sig_pids,
                          excluded_pids=set())
    exclude(exp, ai_pids + failed_pids + incomplete_pids + high_miss_pids)
    print(f'Included n = {len(exp.included_pids)}')
    return exp

In [ ]:
### FIGURES 

def _draw_panels(panels, plot_fn, title, panel_size, figures_dir, fname, sharey, show=None, **kwargs):
    """One row of panels, panels = [(data, panel_title)]; with pilot data (USE_PILOT / COMPARE_PILOT) every title
    gets its n on a second line.
    A lone untitled panel takes the figure title itself; otherwise it goes above the row. Legend on the last panel."""
    w, h = panel_size
    fig, axes = plt.subplots(1, len(panels), figsize=(w * len(panels), h), sharey=sharey, squeeze=False)
    axes = axes[0]
    ylim = kwargs.pop('ylim', None)  # set once every panel is drawn: with a shared y, setting it early freezes it
    legend_opts = kwargs.pop('legend_opts', {})  # e.g. dict(loc='lower right', fontsize=10) pins the legend
    n_line = lambda n: f'\n(n={n})' if USE_PILOT or COMPARE_PILOT else ''
    for i, (ax, (panel_data, panel_title)) in enumerate(zip(axes, panels)):
        n = plot_fn(ax, panel_data, **kwargs)
        if panel_title is not None:
            ax.set_title(panel_title + n_line(n))
        if sharey and i > 0:
            ax.set_ylabel('')
        if ax.get_legend() is not None:
            if i < len(axes) - 1:
                ax.get_legend().remove()
    if ylim is not None and ylim != (None, None):
        for ax in axes:
            ax.set_ylim(*ylim)
    solo_title = textwrap.fill(title, int(w * 7)) + n_line(n)  # ~7 title chars per inch
    if len(panels) == 1 and panels[0][1] is None:
        axes[0].set_title(solo_title)
    else:
        fig.suptitle(title)
    # resize so each axes matches a solo panel's (w x h, legend aside): inner shared-y panels drop the y label/ticks
    # (width), the suptitle + panel titles take more room than the solo title (height), and the legend adds width
    plt.tight_layout()
    renderer = fig.canvas.get_renderer()
    first, last = axes[0].get_window_extent(renderer), axes[-1].get_window_extent(renderer)
    ignored = [a for a in (axes[-1].get_legend(), axes[0].title, axes[-1].title) if a is not None]  # not fixed margin
    for artist in ignored:
        artist.set_visible(False)
    overhang = (first.x0 - axes[0].get_tightbbox(renderer).x0 + axes[-1].get_tightbbox(renderer).x1 - last.x1) / fig.dpi
    for artist in ignored:
        artist.set_visible(True)
    pad = 1.08 * plt.rcParams['font.size'] / 72  # tight_layout's default edge pad, in inches
    solo_width = w - overhang - 2 * pad
    top = fig.get_figheight() - first.y1 / fig.dpi - pad
    panel_title = axes[0].get_title()
    axes[0].set_title(solo_title)
    solo_top = (axes[0].get_tightbbox(renderer).y1 - first.y1) / fig.dpi
    axes[0].set_title(panel_title)
    fig.set_size_inches(fig.get_figwidth() + len(axes) * (solo_width - first.width / fig.dpi),
                        fig.get_figheight() + top - solo_top)
    if axes[-1].get_legend() is not None:  # inside the last panel
        legend_opts = dict(legend_opts)
        _place_legend_inside(fig, axes[-1], [legend_opts.pop('loc')] if 'loc' in legend_opts else LEGEND_LOCS,
                             **legend_opts)
    finish_fig(figures_dir, fname, show=show)


LEGEND_LOCS = ['upper left', 'upper right', 'lower right', 'lower left',
               'center left', 'center right', 'upper center', 'lower center']


def _place_legend_inside(fig, ax, locs=LEGEND_LOCS, grow=1.1, max_tries=10, allow_overlap=False, **legend_kws):
    """Put ax's legend at the first of locs clear of every line, marker and error bar. If none is clear, enlarge
    the figure (text stays the same size, so the data spreads out under the legend) and try again; with
    allow_overlap, keep the size and let matplotlib pick the least-overlapping spot (or keep a single given loc)."""
    renderer = fig.canvas.get_renderer()
    for _ in range(max_tries):
        plt.tight_layout()
        points = _data_points(ax)
        for loc in locs:
            sns.move_legend(ax, loc, **legend_kws)
            box = ax.get_legend().get_window_extent(renderer).padded(6 * fig.dpi / 72)  # ~marker radius of room
            if not box.count_contains(points):
                return
        if allow_overlap:
            sns.move_legend(ax, locs[0] if len(locs) == 1 else 'best', **legend_kws)
            return
        fig.set_size_inches(fig.get_size_inches() * grow)
    print(f'legend still overlaps the data after growing the figure {grow}x{max_tries}: {ax.get_title()!r}')


def _data_points(ax, per_segment=50):
    """Display coords densely along every line (error bars included) and at every scatter point of ax."""
    points = []
    for line in ax.lines:
        xy = line.get_transform().transform(line.get_xydata())
        xy = xy[np.isfinite(xy).all(axis=1)]
        points.append(xy)
        for a, b in zip(xy[:-1], xy[1:]):
            points.append(a + np.linspace(0, 1, per_segment)[:, None] * (b - a))
    for coll in ax.collections:
        points.append(coll.get_offset_transform().transform(coll.get_offsets()))
    return np.concatenate([p for p in points if len(p)]) if points else np.empty((0, 2))


def plot_all_vs_included(sources, figures_dir, plot_fn, title, fname, figsize=(9, 4), included_figsize=None, main=False, **kwargs):
    """For each source (exp, data, label): everyone except suspected AI, then the included sample."""
    panels = []
    for exp, data, label in sources:
        prefix = f'{label}: ' if label else ''
        panels += [(data[~data['participant_id'].isin(exp.ai_pids)], prefix + 'All Participants (excl. AI)'),
                   (data[data['participant_id'].isin(exp.included_pids)], prefix + 'Included')]
    _draw_panels(panels, plot_fn, title, (figsize[0] / 2, figsize[1]), figures_dir, fname,
                 sharey=False, **kwargs)


def plot_included(sources, figures_dir, plot_fn, title, fname, figsize=(9, 4), included_figsize=None, main=False,
                  panel_by=None, panel_order=None, panel_titles=None, **kwargs):
    """Included sample only, saved as <fname>_included.png: one panel per source (exp, data, label), or with
    panel_by one panel per panel_order level for each source. All panels share the y axis."""
    panels = []
    for exp, data, label in sources:
        data = data[data['participant_id'].isin(exp.included_pids)]
        if panel_by is None:
            panels.append((data, label))
        else:
            panels += [(data[data[panel_by] == level], f'{label}\n{panel_title}' if label else panel_title)
                       for level, panel_title in zip(panel_order, panel_titles)]
    size = included_figsize or (max(figsize[0] / 2, 5), figsize[1])
    _draw_panels(panels, plot_fn, title, size, figures_dir, fname.replace('.png', '_included.png'),
                 sharey=True, show=True, **kwargs)


def plot_figures(build_figures, exp, pilot=None):
    """build_figures(exp) -> figure specs, each laid over FIG_DEFAULTS. MAIN_FIGURES_ONLY: just the included panel
    of specs marked main=True; otherwise every spec, both ways. With COMPARE_PILOT, each spec is also built from the
    pilot's data and drawn to the left ('Pilot' | 'Replication'); nothing is drawn without a pilot to compare."""
    if COMPARE_PILOT and pilot is None:
        return
    figures = build_figures(exp)
    compare = COMPARE_PILOT
    pilot_figures = build_figures(pilot) if compare else [None] * len(figures)
    for fig, pilot_fig in zip(figures, pilot_figures):
        fig = {**FIG_DEFAULTS, **fig}
        data = fig.pop('data')
        sources = [(exp, data, None)]
        if compare:
            sources = [(pilot, pilot_fig['data'], 'Pilot'), (exp, data, 'Replication')]
        if not MAIN_FIGURES_ONLY and 'panel_by' not in fig:  # paneled figures are included-only
            plot_all_vs_included(sources, exp.figures_dir, **fig)
        if fig.get('main') or not MAIN_FIGURES_ONLY:
            plot_included(sources, exp.figures_dir, **fig)


# axis options any plot spec can set; plot functions route these to _style_ax instead of seaborn
STYLE_KEYS = ('xlabel', 'ylabel', 'xlim', 'ylim', 'xticks', 'xticklabels', 'yticks', 'yticklabels', 'ytick_step')


def _pop_style(kwargs):
    return {k: kwargs.pop(k) for k in STYLE_KEYS if k in kwargs}


def _style_ax(ax, chance=None, chance_color='gray', xlabel=None, ylabel=None, xlim=None, ylim=None,
              xticks=None, xticklabels=None, yticks=None, yticklabels=None, ytick_step=0.1):
    if chance is not None:
        ax.axhline(chance, linestyle='dashed', color=chance_color)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if xticks is not None:
        ax.set_xticks(xticks, labels=xticklabels)
    if yticks is not None:
        ax.set_yticks(yticks, labels=yticklabels)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None and ylim != (None, None):  # set_ylim freezes autoscaling, even for a None end
        ax.set_ylim(*ylim)
    if yticks is None and np.ptp(ax.get_ylim()) <= 2:  # probabilities and RTs in s: ticks on multiples of ytick_step
        ax.yaxis.set_major_locator(MultipleLocator(ytick_step))


def _markers_on_top(ax):
    """Draw the point markers (and their edges) over the error bars."""
    for line in ax.lines:
        if line.get_marker() not in (None, '', 'None'):
            line.set_zorder(3)


def plot_subject_lines(ax, data, x, y, order=None, color='steelblue', errorbar=('ci', 95), chance=0.5, chance_color='gray', **kwargs):
    """Per-subject means as light gray lines, with the group mean +/- SE on top."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x], as_index=False).agg({y: 'mean'})
    if order is not None:
        # otherwise the lineplot fixes the x order by first appearance, ignoring `order`
        plot_df[x] = pd.Categorical(plot_df[x], categories=order, ordered=True)
    sns.lineplot(x=x, y=y, units='participant_id', estimator=None, data=plot_df, color='lightgray', ax=ax)
    sns.pointplot(x=x, y=y, data=plot_df, errorbar=errorbar, color=color, order=order, ax=ax, **kwargs)
    _markers_on_top(ax)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()


def plot_hue_means(ax, data, x, y, hue, order=None, hue_order=None, palette=None, hue_labels=None,
                   legend_title=None, dodge=0.3, errorbar=('ci', 95), extra_by=(),
                   chance=None, chance_color='gray', **kwargs):
    """Group mean +/- error bar of per-subject means, split by hue. extra_by: extra columns to
    average within before taking each subject's mean."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x, hue, *extra_by], as_index=False).agg({y: 'mean'})
    sns.pointplot(x=x, y=y, hue=hue, data=plot_df, errorbar=errorbar, order=order, hue_order=hue_order,
                  palette=palette, dodge=dodge, ax=ax, **kwargs)
    _markers_on_top(ax)
    if hue_labels is not None or legend_title is not None:
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles=handles, labels=hue_labels or labels, title=legend_title)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()


def plot_colored_means(ax, data, x, y, order, palette, errorbar=('ci', 95), line_color='gray',
                       split_by=None, split_order=None, split_palettes=None, split_line_colors=None,
                       split_labels=None, legend_title=None, dodge=0.08,
                       chance=None, chance_color='gray', **kwargs):
    """Group mean +/- error bar of per-subject means, one color per x level, joined by a plain line.
    split_by: one such line per split_order level, each with its own palette / line color, dodged apart."""
    style = _pop_style(kwargs)
    plot_df = data.groupby(['participant_id', x, *([split_by] if split_by else [])], as_index=False).agg({y: 'mean'})
    if split_by is None:
        lines = [(plot_df, palette, line_color, 0.0)]
    else:
        offsets = np.linspace(-dodge, dodge, len(split_order))
        lines = [(plot_df[plot_df[split_by] == level], pal, lc, off)
                 for level, pal, lc, off in zip(split_order, split_palettes, split_line_colors, offsets)]
    for line_df, pal, lc, off in lines:
        n_before = len(ax.lines)
        sns.pointplot(x=x, y=y, hue=x, data=line_df, order=order, hue_order=order, palette=pal,
                      errorbar=errorbar, legend=False, ax=ax, **kwargs)
        for ln in ax.lines[n_before:]:
            ln.set_xdata(np.asarray(ln.get_xdata(), dtype=float) + off)
        means = line_df.groupby(x)[y].mean().reindex(order)
        ax.plot(np.arange(len(order)) + off, means.values, color=lc, linewidth=2, zorder=1)
    if split_by is not None:
        marker_kws = {k: v for k, v in kwargs.items() if k.startswith('marker')}
        handles = [Line2D([], [], color=lc, marker='o', markerfacecolor=lc, **marker_kws) for lc in split_line_colors]
        ax.legend(handles=handles, labels=split_labels, title=legend_title)
    _markers_on_top(ax)
    _style_ax(ax, chance, chance_color, **style)
    return plot_df['participant_id'].nunique()

In [ ]:
### UTILITIES (unused)

def print_bonuses(df):
    """participant_id,final_bonus for bonus payment."""
    bonus_df = df.groupby('participant_id', as_index=False).agg({'final_bonus': 'max'}).sort_values('participant_id')
    print('For bonus payment: ')
    for _, row in bonus_df.iterrows():
        print(f'{row.participant_id},{row.final_bonus}')


def plot_rt_distributions(df, figures_dir, col_wrap=10):
    g = sns.displot(x='rt', data=df.query('is_choice_trial == True'), col='participant_id', col_wrap=col_wrap)
    g.figure.suptitle(f"All Choice Trials\n(n={df['participant_id'].nunique()})", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

    g = sns.displot(x='rt', data=df.query('old_trial == 1'), col='participant_id', col_wrap=col_wrap)
    g.figure.suptitle(f"Old Trials\n(n={df['participant_id'].nunique()})", fontsize=16, y=1.02)
    finish_fig(figures_dir, 'rt_dist.png')

# Experiment 1 (Main)

In [ ]:
exp1 = setup_experiment('main')
exp1_pilot = setup_experiment('main', data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp1.df)

In [ ]:
no_yes = dict(xticks=[0, 1], xticklabels=['No', 'Yes'])
value_ticks = dict(xticks=[0, 1], xticklabels=['$0', '$1'], xlim=(-0.2, 1.2))
EXP1_FIGSIZE = (3.35, 4.55)  # exp1 main accuracy + RT (smallest size their legends clear the data at); exp2's main plots match


def exp1_figures(exp):
    old_trials = exp.df.query('old_trial == 1').dropna(subset=['optimal_old_choice']).copy()
    old_trials['rt_sec'] = old_trials['rt'] / 1000
    return [
        dict(data=old_trials, plot_fn=plot_subject_lines, title='Old Trial Choices', fname='optimal.png',
             x='old_value', y='old_chosen', color=MEM_PALETTES['blue']['mid'], native_scale=True, **value_ticks,
             xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(data=old_trials, plot_fn=plot_hue_means, title='Old Trial Choices by Delay', fname='optimal_delay.png',
             included_figsize=(5, 4), figsize=(10, 4), x='delay', y='old_chosen', hue='old_value',
             hue_order=[0.0, 1.0], palette=VALUE_PALETTE, hue_labels=['$0', '$1'], legend_title='Old Card Value',
             dodge=False, **marker_style(2), chance=0.5, xlabel='Encoding-Retrieval Delay',
             ylabel='P(Choose Old Card)'),
        dict(data=old_trials, plot_fn=plot_hue_means, title='Old Trial Choices',
             fname='optimal_mem.png', ylim=(0.3, 0.8), main=True, figsize=(10, 4), x='old_value', y='old_chosen',
             hue='memorability_bin', hue_order=MEM_ORDER, dodge=True, palette=MEM_PALETTES['blue'], **marker_style(3),
             native_scale=True, chance=0.5, **value_ticks, hue_labels=MEM_LABELS, legend_title='Memorability Bin',
             included_figsize=EXP1_FIGSIZE, legend_opts=dict(loc='lower right', fontsize=10, title_fontsize=10, grow=1.02),
             xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(data=old_trials, plot_fn=plot_subject_lines, title='Old Trial RT', fname='rt.png', figsize=(10, 4),
             x='optimal_old_choice', y='rt_sec', color=MEM_PALETTES['purple']['mid'], chance=None, **no_yes,
             xlabel='Optimal Choice', ylabel='RT (s)'),
        dict(data=old_trials, plot_fn=plot_hue_means, title='Old Trial RT', fname='rt_mem.png',
             ylim=(None, None), ytick_step=0.05, main=True, figsize=(10, 4), x='optimal_old_choice', y='rt_sec',
             hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'], **marker_style(3),
             extra_by=['old_value'], dodge=True, hue_labels=MEM_LABELS, legend_title='Memorability Bin',
             xticks=[0, 1], xticklabels=['Incorrect', 'Correct'],
             included_figsize=EXP1_FIGSIZE, legend_opts=dict(fontsize=10, title_fontsize=10, grow=1.02),
             xlabel='Choice', ylabel='RT (s)'),
    ]


plot_figures(exp1_figures, exp1, exp1_pilot)


# TEMP RT workshop: current plot | correct vs incorrect | chose old x correct -- delete once we pick one
rt_df = exp1.df.query('old_trial == 1').dropna(subset=['optimal_old_choice']).copy()
rt_df = rt_df[rt_df['participant_id'].isin(exp1.included_pids)]
rt_df['rt_sec'] = rt_df['rt'] / 1000
rt_df['correct_mem'] = rt_df['optimal_old_choice'].map({0: 'incorrect', 1: 'correct'}) + '_' + rt_df['memorability_bin'].astype(str)
fig, axes = plt.subplots(1, 3, figsize=(10.5, 4.8), sharey=True)
mem_kws = dict(y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'],
               **marker_style(3), extra_by=['old_value'], dodge=True, hue_labels=MEM_LABELS,
               legend_title='Memorability Bin', ylabel='RT (s)')
plot_hue_means(axes[0], rt_df, x='old_chosen', **mem_kws, **no_yes, xlabel='Chose Old Card')
plot_hue_means(axes[1], rt_df, x='optimal_old_choice', **mem_kws, xticks=[0, 1],
               xticklabels=['Incorrect', 'Correct'], xlabel='Choice')
plot_hue_means(axes[2], rt_df, x='old_chosen', y='rt_sec', hue='correct_mem',
               hue_order=[f'{c}_{m}' for c in ('incorrect', 'correct') for m in MEM_ORDER],
               palette={**{f'incorrect_{m}': GRAY_PALETTE[m] for m in MEM_ORDER},
                        **{f'correct_{m}': MEM_PALETTES['purple'][m] for m in MEM_ORDER}},
               **marker_style(3), dodge=0.5, legend_title='Memorability Bin',
               hue_labels=[f'{l}, {c}' for c in ('Incorrect', 'Correct') for l in MEM_LABELS], **no_yes,
               xlabel='Chose Old Card')
for ax, t in zip(axes, ['Old Card Chosen', 'Correct vs. Incorrect', 'Both']):
    ax.set_title(t)
    ax.legend(handles=ax.get_legend().legend_handles, labels=[x.get_text() for x in ax.get_legend().texts],
              title='Memorability Bin', fontsize=8, title_fontsize=8, loc='lower left')
for ax in axes[1:]:
    ax.set_ylabel('')
axes[1].get_legend().remove()  # same key as the first panel
fig.suptitle('Old Trial RT')
plt.tight_layout()
plt.show()


# Experiment 2 (Direct Recall)

The direct task has no old/new choice on repeated-card trials -- it has two separate memory tests instead:
recognition (did you pick the seen-before card?) and a value report (what was it worth?). Recognition accuracy
(analogous to `optimal_old_choice` elsewhere) defines the per-subject significance / inclusion set.

In [ ]:
DIRECT_SETUP = dict(label='direct', perf_query='is_recognition_trial == True', perf_col='recognition_correct',
                    perf_label='Recognition Accuracy', min_trials=100)
exp2 = setup_experiment(**DIRECT_SETUP)
exp2_pilot = setup_experiment(**DIRECT_SETUP, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp2.df)

In [ ]:
def load_direct_trials(exp):
    """Recognition and value-report trials, with their outcome columns made numeric."""
    recog_df = exp.df.query('is_recognition_trial == True').copy()
    recog_df['recognition_correct'] = pd.to_numeric(recog_df['recognition_correct'], errors='coerce')
    recog_df['choice_missed'] = pd.to_numeric(recog_df['choice_missed'], errors='coerce')

    value_df = exp.df.query('is_value_test_trial == True').copy()
    value_df['value_test_correct'] = pd.to_numeric(value_df['value_test_correct'], errors='coerce')
    value_df['value_test_missed'] = pd.to_numeric(value_df['value_test_missed'], errors='coerce')
    value_df['old_value'] = pd.to_numeric(value_df['old_value'], errors='coerce')
    return recog_df, value_df


# A subset of sessions (mostly Windows/Chrome) have a bug where keyboard responses on the
# value-report screen intermittently fail to register, while recognition-phase responses stay normal.
# The preregistered > 20% miss-rate exclusion (across all response trials) is applied in setup_experiment;
# this just shows miss rates per phase.
MISS_THRESHOLD = 0.2


def plot_miss_rates(exp):
    recog_df, value_df = load_direct_trials(exp)
    recog_miss_by_subj = recog_df.groupby('participant_id')['choice_missed'].mean()
    value_miss_by_subj = value_df.groupby('participant_id')['value_test_missed'].mean()

    # miss-rate histograms over ALL participants, to show the bug's true prevalence
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    fig.suptitle(f'Miss Rate per Participant, by Trial Phase ({exp.source})')
    for ax, miss, color, name in [(axes[0], recog_miss_by_subj, 'steelblue', 'Recognition-phase'),
                                  (axes[1], value_miss_by_subj, 'tomato', 'Value-recall')]:
        ax.hist(miss, bins=20, range=(0, 1), color=color, edgecolor='white')
        ax.axvline(MISS_THRESHOLD, linestyle='dashed', color='red')
        ax.set_title(f'{name} miss rate\n(n={miss.shape[0]})')
        ax.set_xlabel('P(missed)')
        ax.set_ylabel('Number of participants')
    finish_fig(exp.figures_dir, 'direct_miss_rate_histograms.png')


if not COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
    plot_miss_rates(exp2)

In [ ]:
mem_ticks = dict(xticks=[0, 1, 2], xticklabels=MEM_LABELS, xlabel='Memorability Bin')
correct_ticks = dict(xticks=[0, 1], xticklabels=['Incorrect', 'Correct'])
hit_miss_panels = dict(panel_by='recognition_hit', panel_order=[1.0, 0.0], panel_titles=['Hit Trials', 'Miss Trials'])


def exp2_figures(exp):
    recog_df, value_df = load_direct_trials(exp)
    recog_valid = recog_df.dropna(subset=['recognition_correct', 'rt']).astype({'recognition_correct': int})
    recog_valid['rt_sec'] = recog_valid['rt'] / 1000
    # each value report follows its recognition trial (same trial_number); hit = recognized correctly
    value_df = value_df.merge(recog_df[['participant_id', 'trial_number', 'recognition_correct']]
                              .rename(columns={'recognition_correct': 'recognition_hit'}),
                              on=['participant_id', 'trial_number'], how='left')
    value_valid = value_df.dropna(subset=['value_test_correct', 'rt']).astype({'value_test_correct': int})
    value_valid['rt_sec'] = value_valid['rt'] / 1000

    return [
        dict(data=recog_df, plot_fn=plot_colored_means, title='Old Trial Recognition', main=True,
             fname='direct_recognition_by_mem.png', included_figsize=EXP1_FIGSIZE, ylim=(None, 1.0), x='memorability_bin', y='recognition_correct',
             order=MEM_ORDER, palette=MEM_PALETTES['green'], chance=0.5, **mem_ticks,
             ylabel='Accuracy'),
        dict(data=value_df, plot_fn=plot_colored_means, title='Value Recall', main=True,
             fname='direct_value_accuracy_by_mem.png', included_figsize=EXP1_FIGSIZE, ylim=(None, 0.7), x='memorability_bin', y='value_test_correct',
             order=MEM_ORDER, palette=MEM_PALETTES['blue'], chance=0.5, **mem_ticks,
             ylabel='Accuracy'),
        dict(data=value_df, plot_fn=plot_hue_means,
             title='Value-Recall Accuracy, split by True Card Value',
             fname='direct_value_accuracy_by_mem_and_value.png', included_figsize=(5, 4), x='memorability_bin',
             y='value_test_correct', hue='old_value', order=MEM_ORDER, hue_order=[0.0, 1.0], palette=VALUE_PALETTE,
             **marker_style(2), hue_labels=['$0', '$1'], legend_title='Old Card Value',
             chance=0.5, **mem_ticks, ylabel='Accuracy', ylim=(0.35, 0.8),
             yticks=[0.4, 0.5, 0.6, 0.7, 0.8]),
        dict(data=recog_valid, plot_fn=plot_hue_means, title='Recognition RT', main=True,
             fname='direct_recognition_rt.png', included_figsize=EXP1_FIGSIZE, ylim=(None, 1.1), ytick_step=0.05, x='recognition_correct',
             legend_opts=dict(loc='upper right', fontsize=10, title_fontsize=10, allow_overlap=True),
             y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'],
             **marker_style(3), dodge=True, hue_labels=MEM_LABELS, legend_title='Memorability Bin', **correct_ticks,
             xlabel='Recognition', ylabel='RT (s)'),
        dict(data=value_valid, plot_fn=plot_hue_means, title='Value-Recall RT', main=True,
             fname='direct_value_rt.png', included_figsize=EXP1_FIGSIZE, ylim=(None, 0.75), ytick_step=0.05, x='value_test_correct', y='rt_sec',
             legend_opts=dict(loc='upper right', fontsize=10, title_fontsize=10, allow_overlap=True),
             hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['purple'], **marker_style(3),
             dodge=True, hue_labels=MEM_LABELS, legend_title='Memorability Bin', **correct_ticks,
             xlabel='Value Recall', ylabel='RT (s)'),
        dict(data=value_df, plot_fn=plot_colored_means, title='Value Recall', main=True,
             fname='direct_value_accuracy_by_mem_hit_miss.png', ylim=(None, 0.7), **hit_miss_panels,
             x='memorability_bin', y='value_test_correct', order=MEM_ORDER, palette=MEM_PALETTES['blue'], chance=0.5,
             **mem_ticks, ylabel='Accuracy'),
        dict(data=value_valid, plot_fn=plot_hue_means, title='Value-Recall RT', main=True,
             fname='direct_value_rt_hit_miss.png', ylim=(None, None), **hit_miss_panels,
             x='value_test_correct', y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER,
             palette=MEM_PALETTES['purple'], **marker_style(3), dodge=True, hue_labels=MEM_LABELS,
             legend_title='Memorability Bin', **correct_ticks, xlabel='Value Recall', ylabel='RT (s)'),
    ]

In [ ]:
plot_figures(exp2_figures, exp2, exp2_pilot)

# Experiment 3 (Matched-Memorability)

In [ ]:
exp3 = setup_experiment('matched_memorability', min_trials=70)
exp3_pilot = setup_experiment('matched_memorability', min_trials=70, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp3.df)

In [ ]:
MATCHED_MEM_ORDER = ['low', 'high']

matched_ticks = dict(xticks=[0, 1], xticklabels=['Low', 'High'], xlim=(-0.5, 1.5), xlabel='Memorability Bin')
# incorrect in gray, correct in the measure's color; both light -> dark for low -> high memorability
correct_split = dict(split_by='optimal_old_choice', split_order=[0.0, 1.0], split_labels=['Incorrect', 'Correct'],
                     split_palettes=[GRAY_PALETTE, MEM_PALETTES['purple']],
                     split_line_colors=[GRAY_PALETTE['mid'], MEM_PALETTES['purple']['mid']])


def exp3_figures(exp):
    old_matched = exp.df.query('old_trial == 1').copy()
    old_matched['optimal_old_choice'] = pd.to_numeric(old_matched['optimal_old_choice'], errors='coerce')
    old_matched['chosen_value'] = pd.to_numeric(old_matched['chosen_value'], errors='coerce')
    old_matched['rt_sec'] = old_matched['rt'] / 1000
    return [
        dict(data=old_matched.dropna(subset=['optimal_old_choice']), plot_fn=plot_colored_means, main=True,
             title='Old Trial Choices', fname='matched_optimal_by_mem.png', ylim=(None, 0.8),
             x='memorability_bin', y='optimal_old_choice', order=MATCHED_MEM_ORDER, palette=MEM_PALETTES['blue'],
             chance=0.5, **matched_ticks, ylabel='P(Optimal Choice)'),
        dict(data=old_matched.dropna(subset=['optimal_old_choice']), plot_fn=plot_colored_means, main=True,
             title='Old Trial RT', fname='matched_rt_by_mem.png', ylim=(1.0, None), ytick_step=0.05,
             legend_opts=dict(loc='lower left', allow_overlap=True),
             x='memorability_bin', y='rt_sec', order=MATCHED_MEM_ORDER, palette=None, **correct_split,
             **marker_style(2), **matched_ticks, ylabel='RT (s)'),
        dict(data=old_matched, plot_fn=plot_hue_means, title='RT by Memorability and Chosen Value',
             fname='matched_rt_by_mem_and_value.png', included_figsize=(6.5, 5), ytick_step=0.05, figsize=(13, 5),
             x='memorability_bin', y='rt_sec', hue='chosen_value', order=MATCHED_MEM_ORDER, hue_order=[0.0, 1.0],
             palette=VALUE_PALETTE, **marker_style(2), hue_labels=['$0', '$1'], legend_title='Chose', **matched_ticks,
             ylabel='RT (s)'),
    ]


plot_figures(exp3_figures, exp3, exp3_pilot)

# Experiment 4 (Mixed-Memorability)

df.ret_type:
- 1 -> high=0, low=1
- 2 -> high=1, low=0
- 3 -> high=0, low=0
- 4 -> high=1, low=1

In [ ]:
# significance is on optimal_old_choice, which is only defined on the uneven (ret_type 1/2) trials
exp4 = setup_experiment('mixed_memorability', min_trials=70)
exp4_pilot = setup_experiment('mixed_memorability', min_trials=70, data_root=PILOT_ROOT) if COMPARE_PILOT else None
# print_bonuses(exp4.df)

In [ ]:
# Human-readable labels & display order
RT_LABELS = {1: 'High=\\$0\nLow=\\$1', 2: 'High=\\$1\nLow=\\$0', 3: 'Both=\\$0', 4: 'Both=\\$1'}
RT_ORDER     = [RT_LABELS[k] for k in (1, 2, 3, 4)]
UNEVEN_ORDER = RT_ORDER[:2]
EVEN_ORDER   = RT_ORDER[2:]


def plot_mem_bias_all_types(ax, data, **kwargs):
    n = plot_subject_lines(ax, data, 'ret_type_label', 'chose_high_mem', **{**kwargs, **dict(
        order=RT_ORDER, color=VALUE_PALETTE[1.0], xlabel='Retrieval Type', ylabel='P(Choose Memorable)', ylim=(0, 1))})
    # shade uneven vs even region
    ax.axvspan(-0.5, 1.5, alpha=0.06, color='gray', label='Uneven value')
    ax.axvspan(1.5, 3.5, alpha=0.06, color=VALUE_PALETTE[0.0], label='Even value')
    ax.legend(fontsize=8)
    return n


value_ticks = dict(xticks=[0, 1], xticklabels=['$0', '$1'], xlim=(-0.5, 1.5))
uneven_palette = dict(zip(UNEVEN_ORDER, VALUE_PALETTE.values()))
even_palette = dict(zip(EVEN_ORDER, VALUE_PALETTE.values()))
uneven_ticks = dict(xticks=[0, 1], xticklabels=['High=\\$0\nLow=\\$1', 'High=\\$1\nLow=\\$0'], xlim=(-0.5, 1.5))


def exp4_figures(exp):
    old_mixed = exp.df.query('old_trial == 1').copy()
    old_mixed['optimal_old_choice'] = pd.to_numeric(old_mixed['optimal_old_choice'], errors='coerce')
    # Which memorability bin did the participant effectively choose?
    old_mixed['chose_high_mem'] = (
        ((old_mixed['chosen_side'] == 'left')  & (old_mixed['left_mem_bin']  == 'high')) |
        ((old_mixed['chosen_side'] == 'right') & (old_mixed['right_mem_bin'] == 'high'))
    ).astype(float)
    old_mixed['h_longer_delay'] = (old_mixed['delay_h'] > old_mixed['delay_l']).astype(float)
    old_mixed['rt_sec'] = old_mixed['rt'] / 1000
    old_mixed['ret_type_label'] = old_mixed['ret_type'].map(RT_LABELS)
    uneven_trials = old_mixed.dropna(subset=['optimal_old_choice'])
    even_trials = old_mixed.query('ret_type in [3, 4]')

    return [
        dict(data=even_trials, plot_fn=plot_colored_means, main=True, title='Even Trial Choices',
             fname='mixed_even_mem_bias.png', ylim=(0.35, 0.65), x='ret_type_label', y='chose_high_mem',
             order=EVEN_ORDER, palette=even_palette, chance=0.5, **value_ticks, xlabel='Shared Card Value',
             ylabel='P(Choose Memorable)'),
        # chose the low-mem card in gray, the high-mem card in the $0/$1 oranges
        dict(data=even_trials, plot_fn=plot_colored_means, main=True, title='Even Trial RT',
             fname='mixed_even_rt.png', ylim=(1.0, None), x='ret_type_label', y='rt_sec', order=EVEN_ORDER,
             legend_opts=dict(loc='lower left', fontsize=11, title_fontsize=11, allow_overlap=True),
             palette=None, split_by='chose_high_mem', split_order=[0.0, 1.0], split_labels=['Low-mem', 'High-mem'],
             legend_title='Chose',
             split_palettes=[dict(zip(EVEN_ORDER, [GRAY_PALETTE['low'], GRAY_PALETTE['high']])), even_palette],
             split_line_colors=[GRAY_PALETTE['mid'], VALUE_PALETTE[1.0]], **marker_style(2), **value_ticks,
             ytick_step=0.05, xlabel='Shared Card Value', ylabel='RT (s)'),
        dict(data=uneven_trials, plot_fn=plot_colored_means, main=True, title='Uneven Trial Choices',
             fname='mixed_uneven_optimal.png', ylim=(None, 0.9), x='ret_type_label', y='optimal_old_choice',
             order=UNEVEN_ORDER, palette=uneven_palette, chance=0.5, **uneven_ticks, xlabel='High-Mem Card Value',
             ylabel='Accuracy'),
        # incorrect in gray, correct in the $0/$1 oranges
        dict(data=uneven_trials, plot_fn=plot_colored_means, main=True, title='Uneven Trial RT',
             fname='mixed_uneven_rt.png', ylim=(1.0, None), x='ret_type_label', y='rt_sec', order=UNEVEN_ORDER,
             legend_opts=dict(loc='lower left', allow_overlap=True),
             palette=None, split_by='optimal_old_choice', split_order=[0.0, 1.0],
             split_labels=['Incorrect', 'Correct'],
             split_palettes=[dict(zip(UNEVEN_ORDER, [GRAY_PALETTE['low'], GRAY_PALETTE['high']])), uneven_palette],
             split_line_colors=[GRAY_PALETTE['mid'], VALUE_PALETTE[1.0]], **marker_style(2), **uneven_ticks,
             ytick_step=0.05, xlabel='High-Mem Card Value', ylabel='RT (s)'),
        dict(data=old_mixed, plot_fn=plot_hue_means, title='RT by Retrieval Type and Chosen Memorability Bin',
             fname='mixed_rt_by_type.png', included_figsize=(6.5, 5), ytick_step=0.05, figsize=(13, 5),
             x='ret_type_label', y='rt_sec', hue='chose_high_mem', order=RT_ORDER, hue_order=[0.0, 1.0],
             palette={0.0: GRAY_PALETTE['mid'], 1.0: VALUE_PALETTE[1.0]}, **marker_style(2),
             hue_labels=['Low-mem', 'High-mem'], legend_title='Chose', xlabel='Retrieval Type', ylabel='RT (s)'),
        dict(data=old_mixed, plot_fn=plot_mem_bias_all_types,
             title='Overall Memorability Bias: P(chose high-mem item) Across All Retrieval Types',
             fname='mixed_mem_bias_all_types.png', included_figsize=(6, 4), figsize=(12, 4)),
    ]


plot_figures(exp4_figures, exp4, exp4_pilot)

# Other (pilot)

## THINGS stimulus memorability scores

In [ ]:
stims = pd.read_csv('../stimuli/stimuli_metadata.csv')
things = pd.read_csv('../stimuli/merged_things_metadata.csv')
stims.columns

In [ ]:
if not COMPARE_PILOT:  # comparison runs only make pilot-vs-replication figures
    # sns.histplot(things['memorability_score'])
    plt.figure(figsize=(5,4))
    plt.title('Selected image memorability scores (THINGS)')
    sns.histplot(data = stims, x = 'memorability_score', hue='memorability_bin', hue_order=['low','mid','high'], palette=['tab:blue','tab:orange','tab:green'], alpha=0.7)
    plt.xlabel('Memorability Score')
    plt.yticks([0,50,100,150])
    plt.tight_layout()
    plt.show()

## Graded value pilot

In [ ]:
if not MAIN_FIGURES_ONLY and not COMPARE_PILOT:
    graded = setup_experiment('graded_value', data_root=PILOT_ROOT)

In [ ]:
def graded_figures(exp):
    graded_old = exp.df.query('old_trial == 1').dropna(subset=['optimal_old_choice']).copy()
    graded_old['rt_sec'] = graded_old['rt'] / 1000
    return [
        dict(data=graded_old, plot_fn=plot_subject_lines, title='Old Trial Choices', fname='optimal.png',
             included_figsize=(5, 4), x='old_value', y='old_chosen', color=MEM_PALETTES['blue']['mid'],
             native_scale=True, xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(data=graded_old, plot_fn=plot_hue_means, title='Old Trial Choices',
             fname='optimal_mem.png', figsize=(10, 4), included_figsize=(5, 4), x='old_value', y='old_chosen',
             hue='memorability_bin', hue_order=MEM_ORDER, palette=MEM_PALETTES['blue'], **marker_style(3), dodge=True,
             native_scale=True, chance=0.5, hue_labels=MEM_LABELS, legend_title='Memorability Bin',
             xlabel='Old Card Value', ylabel='P(Choose Old Card)'),
        dict(data=graded_old, plot_fn=plot_hue_means, title='Old Trial RT', fname='rt_mem.png',
             figsize=(10, 4), x='old_chosen', y='rt_sec', hue='memorability_bin', hue_order=MEM_ORDER,
             palette=MEM_PALETTES['purple'], **marker_style(3), extra_by=['old_value'], dodge=True,
             hue_labels=MEM_LABELS, legend_title='Memorability Bin', xticks=[0, 1], xticklabels=['No', 'Yes'],
             ytick_step=0.05, xlabel='Chose Old Card', ylabel='RT (s)'),
    ]


if not MAIN_FIGURES_ONLY and not COMPARE_PILOT:
    plot_figures(graded_figures, graded)